In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ppi_py import ppi_ols_ci, classical_ols_ci

df = pd.read_csv("data/webgpt_sonnet_5k.csv")
mask = df["yhat"].notna()
df = df[mask].reset_index(drop=True)

# Covariate: difference in response length (len(answer_1) - len(answer_0))
len_0 = df["answer_0"].fillna("").str.len().values.astype(float)
len_1 = df["answer_1"].fillna("").str.len().values.astype(float)
length_diff = len_1 - len_0

# Standardize for numerical stability
length_diff_mean = length_diff.mean()
length_diff_std = length_diff.std()
X_raw = (length_diff - length_diff_mean) / length_diff_std

# Design matrix: intercept + standardized length difference
X = np.column_stack([np.ones(len(df)), X_raw])
Y = df["vote"].values
Yhat = df["yhat"].values

# Permute data so the fixed-split table is not biased by row order
rng0 = np.random.default_rng(42)
perm = rng0.permutation(len(Y))
X, Y, Yhat = X[perm], Y[perm], Yhat[perm]

print(f"N = {len(df)}")
print(f"Columns: intercept, standardized length_diff(B-A)")
df[["question", "vote", "yhat"]].head()

N = 5000
Columns: intercept, standardized length_diff(B-A)


,question,vote,yhat
0,"Voiced by Harry Shearer, what Simpsons charact...",0.00,0.200
1,Alliumphobia is the irrational fear of which p...,0.50,0.250
2,Heterophobia is the irrational fear of what,0.25,0.900
3,"What was the name of Dan Dare's co-pilot, in t...",0.50,0.675
4,"In 1965, which Christmas song became the first...",0.50,0.650


In [15]:
# Single-split demo at n=100
n = 100
alpha = 0.05
coord = 1  # the length_diff coefficient

X_l, X_u = X[:n], X[n:]
Y_l, Yh_l, Yh_u = Y[:n], Yhat[:n], Yhat[n:]

classical_ci = classical_ols_ci(X_l, Y_l, alpha=alpha)
ppi_ci = ppi_ols_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=1)
ppi_pp_ci = ppi_ols_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha)

all_cis = [("Classical", classical_ci), ("PPI", ppi_ci), ("PPI++", ppi_pp_ci)]
classical_width = float(classical_ci[1][coord]) - float(classical_ci[0][coord])

print(f"Association coefficient (length_diff -> vote), n={n}")
print()
print(f"{'Method':<12} {'CI':>22}   {'Width':>8}   {'vs Classical':>12}")
print("-" * 60)
for name, ci in all_cis:
    lo, hi = float(ci[0][coord]), float(ci[1][coord])
    w = hi - lo
    reduction = (1 - w / classical_width) * 100
    ci_str = f"[{lo:.4f}, {hi:.4f}]"
    red_str = "---" if name == "Classical" else f"{reduction:+.1f}%"
    print(f"{name:<12} {ci_str:>22}   {w:>8.4f}   {red_str:>12}")

Association coefficient (length_diff -> vote), n=100

Method                           CI      Width   vs Classical
------------------------------------------------------------
Classical          [0.0037, 0.1016]     0.0979            ---
PPI               [-0.0044, 0.1325]     0.1369         -39.9%
PPI++              [0.0081, 0.1034]     0.0952          +2.7%


In [16]:
# Width and coverage as a function of n
alpha = 0.05
n_trials = 200
ns = np.arange(20, 520, 20)
coord = 1  # length_diff coefficient

# "True" coefficient from full-data OLS
beta_true = np.linalg.lstsq(X, Y, rcond=None)[0][coord]
print(f"Full-data OLS coefficient for length_diff: {beta_true:.4f}")

widths = {m: np.zeros(len(ns)) for m in ["Classical", "PPI", "PPI++"]}
covers = {m: np.zeros(len(ns)) for m in ["Classical", "PPI", "PPI++"]}

rng = np.random.default_rng(42)

for j, n in enumerate(ns):
    w_acc = {m: 0.0 for m in widths}
    c_acc = {m: 0 for m in widths}
    for t in range(n_trials):
        idx = rng.permutation(len(Y))
        X_l, X_u = X[idx[:n]], X[idx[n:]]
        Y_l = Y[idx[:n]]
        Yh_l, Yh_u = Yhat[idx[:n]], Yhat[idx[n:]]

        for name, ci in [
            ("Classical", classical_ols_ci(X_l, Y_l, alpha=alpha)),
            ("PPI", ppi_ols_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=1)),
            ("PPI++", ppi_ols_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha)),
        ]:
            lo, hi = float(ci[0][coord]), float(ci[1][coord])
            w_acc[name] += hi - lo
            c_acc[name] += int(lo <= beta_true <= hi)

    for m in widths:
        widths[m][j] = w_acc[m] / n_trials
        covers[m][j] = c_acc[m] / n_trials
    if (j + 1) % 5 == 0:
        print(f"  n={n} done")

Full-data OLS coefficient for length_diff: 0.0483
  n=100 done
  n=200 done
  n=300 done
  n=400 done
  n=500 done


In [17]:
# Table: average CI width and improvement over Classical at each n
rows = []
for j, n in enumerate(ns):
    w_cl = widths["Classical"][j]
    w_ppi = widths["PPI"][j]
    w_pp = widths["PPI++"][j]
    rows.append({
        "n": int(n),
        "Classical": f"{w_cl:.4f}",
        "PPI": f"{w_ppi:.4f}",
        "PPI++ ": f"{w_pp:.4f}",
        "PPI vs Classical": f"{(1 - w_ppi / w_cl) * 100:+.1f}%",
        "PPI++ vs Classical": f"{(1 - w_pp / w_cl) * 100:+.1f}%",
    })

table_df = pd.DataFrame(rows).set_index("n")
print(table_df.to_string())

    Classical     PPI  PPI++  PPI vs Classical PPI++ vs Classical
n                                                                
20     0.2267  0.2503  0.1875           -10.4%             +17.3%
40     0.1638  0.1879  0.1432           -14.7%             +12.6%
60     0.1334  0.1564  0.1190           -17.2%             +10.8%
80     0.1156  0.1337  0.1032           -15.6%             +10.7%
100    0.1054  0.1209  0.0941           -14.7%             +10.7%
120    0.0943  0.1125  0.0858           -19.3%              +9.0%
140    0.0887  0.1045  0.0810           -17.9%              +8.6%
160    0.0823  0.0971  0.0755           -18.0%              +8.2%
180    0.0774  0.0920  0.0707           -18.9%              +8.6%
200    0.0737  0.0875  0.0676           -18.7%              +8.3%
220    0.0700  0.0838  0.0647           -19.7%              +7.5%
240    0.0668  0.0797  0.0613           -19.3%              +8.2%
260    0.0647  0.0772  0.0595           -19.2%              +8.1%
280    0.0

In [ ]:
colors = {"Classical": "#6394EE", "PPI": "#84C87F", "PPI++": "#FF8B00"}
labels = {"Classical": "classical", "PPI": "PPI", "PPI++": "tuned PPI"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Coverage (left)
for m in ["PPI", "Classical", "PPI++"]:
    axes[0].plot(ns, covers[m], label=labels[m], color=colors[m], linewidth=2)
axes[0].axhline(1 - alpha, color="gray", linestyle="dotted", linewidth=1.5)
axes[0].set_xlabel("n", fontsize=14)
axes[0].set_ylabel("")
axes[0].set_title("coverage", fontsize=16)
axes[0].set_ylim(0.55, 1.02)
axes[0].tick_params(axis='both', labelsize=13)
axes[0].yaxis.set_major_locator(plt.MaxNLocator(3))

# Width (right)
for m in ["PPI", "Classical", "PPI++"]:
    axes[1].plot(ns, widths[m], label=labels[m], color=colors[m], linewidth=2)
axes[1].set_xlabel("n", fontsize=14)
axes[1].set_ylabel("")
axes[1].set_title("width", fontsize=16)
axes[1].legend(fontsize=13)
axes[1].tick_params(axis='both', labelsize=13)
axes[1].yaxis.set_major_locator(plt.MaxNLocator(3))

sns.despine(top=True, right=True)

plt.tight_layout()
plt.show()